# Acuerdo entre anotadores y construcción del estándar PURE-300

Entradas: `datasets/anotacion/entregas/anotador_1.csv` y `anotador_2.csv` (los CSV devueltos por los dos anotadores) y `muestra_300_master.csv` (la muestra, con `req_id` y documento). Salidas: `adjudicacion.csv` (plantilla de desacuerdos para el adjudicador) y, cuando la adjudicación está completa, `gold_pure300.csv`.

El coeficiente kappa es el de Cohen (1960): con po la proporción de ítems en que los anotadores coinciden y pc la esperada por azar a partir de los marginales, kappa = (po - pc) / (1 - pc). La interpretación usa los rangos de Landis y Koch (1977), que sus propios autores califican de arbitrarios y que sirven solo como referencia.

## Uso (dos pasadas)

1. Primera pasada, con las dos entregas presentes: valida los archivos, calcula los acuerdos y exporta `adjudicacion.csv` con los ítems en desacuerdo y las columnas vacías `etiqueta_final`, `es_requisito_final` y `justificacion`.
2. El adjudicador, que no participó en la anotación inicial, llena esas columnas a mano con una justificación escrita por ítem. El archivo no incluye el documento de origen, para que decida con la misma información que tuvieron los anotadores: solo el texto.
3. Segunda pasada: valida la adjudicación (ningún ítem sin decidir, ninguna justificación vacía) y construye el estándar. Cada fila registra si su etiqueta viene del acuerdo de los anotadores o de la adjudicación.

El notebook prepara, mide y valida; no propone etiquetas.

In [1]:
import csv
import hashlib
import os
import unicodedata

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from tabulate import tabulate

ANOTACION_DIR = os.path.join("..", "datasets", "anotacion")
ENTREGAS_DIR = os.path.join(ANOTACION_DIR, "entregas")
MASTER_CSV = os.path.join(ANOTACION_DIR, "muestra_300_master.csv")
ADJUDICACION_CSV = os.path.join(ANOTACION_DIR, "adjudicacion.csv")
GOLD_CSV = os.path.join(ANOTACION_DIR, "gold_pure300.csv")

N_ITEMS = 300
COL_ES_REQUISITO = "es_requisito (si/no)"
COL_ETIQUETA = "etiqueta (F o NFR)"
COL_DUDA = "duda (si/no)"
COL_COMENTARIO = "comentario (opcional)"

master = pd.read_csv(MASTER_CSV, dtype=str)
assert len(master) == N_ITEMS

for annotator_file in ("anotador_1.csv", "anotador_2.csv"):
    path = os.path.join(ENTREGAS_DIR, annotator_file)
    assert os.path.exists(path), (
        f"Falta {path}: coloca ahi el CSV devuelto por el anotador antes de correr este notebook")

deliveries = {a: pd.read_csv(os.path.join(ENTREGAS_DIR, f"anotador_{a}.csv"), dtype=str).fillna("")
              for a in (1, 2)}
print(f"Entregas leidas: anotador_1 ({len(deliveries[1])} filas), anotador_2 ({len(deliveries[2])} filas)")

Entregas leidas: anotador_1 (300 filas), anotador_2 (300 filas)


## 1. Validación de las entregas

Se verifica contra el manual: 300 filas, `item` y `texto` idénticos a la muestra y valores dentro del dominio permitido. Las variantes de escritura (mayúsculas, acentos, espacios: "SÍ", "nfr", " F ") se normalizan y cada normalización se reporta. Un problema no normalizable (fila sin etiqueta, texto alterado, filas de más o de menos) detiene el análisis.

In [2]:
def normalize_si_no(value, allow_empty):
    canonical = unicodedata.normalize("NFKD", value.strip().lower())
    canonical = "".join(ch for ch in canonical if not unicodedata.combining(ch))
    if canonical in {"si", "no"}:
        return canonical
    if canonical == "" and allow_empty:
        return ""
    return None

def normalize_etiqueta(value):
    canonical = value.strip().upper()
    return canonical if canonical in {"F", "NFR"} else None

def validate_delivery(annotator_id, delivery):
    normalizations, fatal = [], []
    if len(delivery) != N_ITEMS:
        fatal.append([annotator_id, "-", "filas", f"{len(delivery)} en lugar de {N_ITEMS}"])
        return delivery, normalizations, fatal
    for idx in range(N_ITEMS):
        item = master.loc[idx, "item"]
        if delivery.loc[idx, "item"] != item:
            fatal.append([annotator_id, item, "item", f"'{delivery.loc[idx, 'item']}' fuera de secuencia"])
        if delivery.loc[idx, "texto"] != master.loc[idx, "texto"]:
            fatal.append([annotator_id, item, "texto", "difiere de la muestra congelada"])
        for column, normalizer in ((COL_ES_REQUISITO, lambda v: normalize_si_no(v, allow_empty=False)),
                                   (COL_ETIQUETA, normalize_etiqueta),
                                   (COL_DUDA, lambda v: normalize_si_no(v, allow_empty=True))):
            original = delivery.loc[idx, column]
            normalized = normalizer(original)
            if normalized is None:
                fatal.append([annotator_id, item, column, f"valor no permitido: '{original}'"])
            elif normalized != original:
                normalizations.append([annotator_id, item, column, f"'{original}'", f"'{normalized}'"])
                delivery.loc[idx, column] = normalized
    return delivery, normalizations, fatal

all_normalizations, all_fatal = [], []
for a in (1, 2):
    deliveries[a], normalizations, fatal = validate_delivery(a, deliveries[a])
    all_normalizations += normalizations
    all_fatal += fatal

if all_normalizations:
    print(tabulate(all_normalizations,
                   headers=["anotador", "item", "columna", "original", "normalizado"], tablefmt="github"))
else:
    print("Sin normalizaciones: los valores llegaron exactamente en el formato pedido")

if all_fatal:
    print()
    print(tabulate(all_fatal, headers=["anotador", "item", "columna", "problema"], tablefmt="github"))
    raise ValueError(f"{len(all_fatal)} problema(s) no normalizable(s): revisar la tabla y regresar al anotador")
print(f"\nValidacion superada: 2 entregas x {N_ITEMS} filas, textos identicos a la muestra congelada")

|   anotador |   item | columna      | original   | normalizado   |
|------------|--------|--------------|------------|---------------|
|          1 |      4 | duda (si/no) | 'sí'       | 'si'          |

Validacion superada: 2 entregas x 300 filas, textos identicos a la muestra congelada


## 2. Acuerdo entre anotadores

Kappa de Cohen por separado para `etiqueta` y para `es_requisito`, que son dos decisiones distintas, junto con el porcentaje de acuerdo crudo. Kappa se calcula a mano desde las definiciones (po y pc) y se comprueba contra `sklearn.metrics.cohen_kappa_score`. Después se muestra el acuerdo por documento y su relación con la columna `duda`.

In [3]:
def kappa_by_hand(labels_a, labels_b):
    labels_a, labels_b = np.asarray(labels_a), np.asarray(labels_b)
    categories = sorted(set(labels_a) | set(labels_b))
    po = float(np.mean(labels_a == labels_b))
    pc = sum((np.mean(labels_a == c)) * (np.mean(labels_b == c)) for c in categories)
    return (po - pc) / (1 - pc), po

def landis_koch(kappa):
    for upper, label in ((0.0, "Poor"), (0.20, "Slight"), (0.40, "Fair"),
                         (0.60, "Moderate"), (0.80, "Substantial")):
        if kappa <= upper:
            return label
    return "Almost Perfect"

agreement_rows = []
for column, decision in ((COL_ETIQUETA, "etiqueta"), (COL_ES_REQUISITO, "es_requisito")):
    a1, a2 = deliveries[1][column], deliveries[2][column]
    kappa_manual, po = kappa_by_hand(a1, a2)
    kappa_sklearn = cohen_kappa_score(a1, a2)
    assert abs(kappa_manual - kappa_sklearn) < 1e-12, "kappa manual y sklearn difieren"
    agreement_rows.append([decision, f"{po:.3f}", f"{kappa_manual:.3f}", landis_koch(kappa_manual)])

print(tabulate(agreement_rows,
               headers=["decision", "acuerdo crudo", "kappa de Cohen", "Landis-Koch"], tablefmt="github"))
print('\nInterpretacion segun Landis y Koch (1977, p. 165), cuyos autores advierten que las divisiones')
print('son "clearly arbitrary" (claramente arbitrarias) y sirven solo como puntos de referencia.')

etiqueta_agrees = deliveries[1][COL_ETIQUETA] == deliveries[2][COL_ETIQUETA]
es_req_agrees = deliveries[1][COL_ES_REQUISITO] == deliveries[2][COL_ES_REQUISITO]

by_document = []
for document, idx in master.groupby("documento").groups.items():
    agree_rate = float(etiqueta_agrees[idx].mean())
    by_document.append([document, len(idx), f"{agree_rate:.3f}"])
print()
print(tabulate(sorted(by_document, key=lambda r: r[2]),
               headers=["documento", "items", "acuerdo crudo (etiqueta)"], tablefmt="github"))

any_duda = (deliveries[1][COL_DUDA] == "si") | (deliveries[2][COL_DUDA] == "si")
duda_rows = [["con duda de algun anotador", int(any_duda.sum()),
              f"{float(etiqueta_agrees[any_duda].mean()):.3f}" if any_duda.any() else "-"],
             ["sin duda", int((~any_duda).sum()), f"{float(etiqueta_agrees[~any_duda].mean()):.3f}"]]
print()
print(tabulate(duda_rows, headers=["grupo", "items", "acuerdo crudo (etiqueta)"], tablefmt="github"))

| decision     |   acuerdo crudo |   kappa de Cohen | Landis-Koch   |
|--------------|-----------------|------------------|---------------|
| etiqueta     |           0.703 |            0.385 | Fair          |
| es_requisito |           0.907 |            0.59  | Moderate      |

Interpretacion segun Landis y Koch (1977, p. 165), cuyos autores advierten que las divisiones
son "clearly arbitrary" (claramente arbitrarias) y sirven solo como puntos de referencia.

| documento           |   items |   acuerdo crudo (etiqueta) |
|---------------------|---------|----------------------------|
| 2008 - peering      |      20 |                      0.15  |
| 2008 - keepass      |      30 |                      0.433 |
| 2007-eirene_fun_7-2 |     115 |                      0.696 |
| 0000 - gamma j      |      33 |                      0.697 |
| 0000 - cctns        |      44 |                      0.818 |
| 2007-ertms          |      58 |                      0.966 |

| grupo                      

## 3. Tabla de adjudicación

Los ítems con desacuerdo en `etiqueta` o en `es_requisito` se exportan a `adjudicacion.csv` con las dos anotaciones lado a lado y tres columnas vacías: `etiqueta_final`, `es_requisito_final` y `justificacion`. Si el archivo ya existe, no se regenera, para no sobrescribir una adjudicación hecha.

In [4]:
disagreement_mask = ~(etiqueta_agrees & es_req_agrees)
disagreements = master[disagreement_mask][["item", "texto"]].copy()
for a in (1, 2):
    disagreements[f"a{a}_es_requisito"] = deliveries[a][COL_ES_REQUISITO][disagreement_mask]
    disagreements[f"a{a}_etiqueta"] = deliveries[a][COL_ETIQUETA][disagreement_mask]
    disagreements[f"a{a}_duda"] = deliveries[a][COL_DUDA][disagreement_mask]
    disagreements[f"a{a}_comentario"] = deliveries[a][COL_COMENTARIO][disagreement_mask]

print(f"Items en desacuerdo: {len(disagreements)} de {N_ITEMS} "
      f"({len(disagreements)/N_ITEMS:.1%})")

if os.path.exists(ADJUDICACION_CSV):
    print(f"{ADJUDICACION_CSV} ya existe: no se regenera (borrarlo es decision explicita del adjudicador)")
else:
    template = disagreements.copy()
    template["etiqueta_final"] = ""
    template["es_requisito_final"] = ""
    template["justificacion"] = ""
    template.to_csv(ADJUDICACION_CSV, index=False)
    print(f"Plantilla exportada: {ADJUDICACION_CSV}")
    print("Siguiente paso: el adjudicador llena etiqueta_final, es_requisito_final y justificacion, "
          "y vuelve a correr el notebook completo")

Items en desacuerdo: 95 de 300 (31.7%)
../datasets/anotacion/adjudicacion.csv ya existe: no se regenera (borrarlo es decision explicita del adjudicador)


## 4. Construcción del estándar (segunda pasada)

Solo corre cuando la adjudicación está completa: cada ítem en desacuerdo debe tener `etiqueta_final` en {F, NFR}, `es_requisito_final` en {si, no} y una justificación no vacía. El resultado conserva `req_id` y documento de la muestra y registra en `origen` si la etiqueta viene de acuerdo o de adjudicación.

In [5]:
adjudication = pd.read_csv(ADJUDICACION_CSV, dtype=str).fillna("")
pending = adjudication[(~adjudication["etiqueta_final"].isin(["F", "NFR"]))
                       | (~adjudication["es_requisito_final"].isin(["si", "no"]))
                       | (adjudication["justificacion"].str.strip() == "")]

if len(pending):
    print(f"Adjudicacion incompleta: {len(pending)} de {len(adjudication)} items sin decision valida "
          "(etiqueta_final F/NFR, es_requisito_final si/no y justificacion no vacia).")
    print("El gold no se construye todavia. Items pendientes:", ", ".join(pending["item"].tolist()))
else:
    gold = master.copy()
    gold["etiqueta"] = deliveries[1][COL_ETIQUETA].values
    gold["es_requisito"] = deliveries[1][COL_ES_REQUISITO].values
    gold["origen"] = "acuerdo"
    adjudicated = adjudication.set_index("item")
    for idx in gold.index[disagreement_mask]:
        item = gold.loc[idx, "item"]
        gold.loc[idx, "etiqueta"] = adjudicated.loc[item, "etiqueta_final"]
        gold.loc[idx, "es_requisito"] = adjudicated.loc[item, "es_requisito_final"]
        gold.loc[idx, "origen"] = "adjudicacion"
    gold.to_csv(GOLD_CSV, index=False)
    sha = hashlib.sha256(open(GOLD_CSV, "rb").read()).hexdigest()
    print(f"Gold construido: {GOLD_CSV}")
    print(f"SHA-256 (huella para congelar): {sha}")
    print()
    origin_counts = gold["origen"].value_counts()
    print(tabulate([[k, v] for k, v in origin_counts.items()],
                   headers=["origen de la etiqueta", "items"], tablefmt="github"))

Gold construido: ../datasets/anotacion/gold_pure300.csv
SHA-256 (huella para congelar): 6354e41446fa390433ae98c58a6634640e59cdc2a4d658160685904839e95e2d

| origen de la etiqueta   |   items |
|-------------------------|---------|
| acuerdo                 |     205 |
| adjudicacion            |      95 |


## 5. Estadísticas del estándar

Distribución por documento y clase. Los ítems con `es_requisito = no` forman parte del archivo, con su etiqueta, pero la evaluación del nivel 3 usa solo los que sí son requisitos (265).

In [6]:
if os.path.exists(GOLD_CSV):
    gold = pd.read_csv(GOLD_CSV, dtype=str)
    per_document = []
    for document, subset in gold.groupby("documento"):
        per_document.append([document, len(subset),
                             int((subset["etiqueta"] == "F").sum()),
                             int((subset["etiqueta"] == "NFR").sum()),
                             int((subset["es_requisito"] == "no").sum())])
    per_document.append(["TOTAL", len(gold),
                         int((gold["etiqueta"] == "F").sum()),
                         int((gold["etiqueta"] == "NFR").sum()),
                         int((gold["es_requisito"] == "no").sum())])
    print(tabulate(per_document,
                   headers=["documento", "items", "F", "NFR", "no requisito"], tablefmt="github"))
else:
    print("Aun no hay gold: completar la adjudicacion y volver a correr el notebook")

| documento           |   items |   F |   NFR |   no requisito |
|---------------------|---------|-----|-------|----------------|
| 0000 - cctns        |      44 |   4 |    40 |              1 |
| 0000 - gamma j      |      33 |   7 |    26 |              3 |
| 2007-eirene_fun_7-2 |     115 |  82 |    33 |             16 |
| 2007-ertms          |      58 |  56 |     2 |              0 |
| 2008 - keepass      |      30 |  17 |    13 |              0 |
| 2008 - peering      |      20 |   2 |    18 |             15 |
| TOTAL               |     300 | 168 |   132 |             35 |
